# Technique B geometrification, then conformal 2D mesh

Tracked as `confMesh2d_technique_b.ipynb`.

**Technique B** builds grain polygons from an MCGS labelled image by:

1. placing hybrid Voronoi seeds (`grid_ops.generate_constrained_hybrid_seeds`)
2. unioning cells that share an LFI label (`GrainManifold2D.by_tessellation`)
3. Taubin-smoothing interfaces and trimming to the RVE

The polygons are then meshed with `gsmesh2d.mesh_gs` (raw Gmsh). `confMesh2d` (pygmsh) is deprecated.

Neighbour / junction bookkeeping that used to live as notebook-local loops is omitted: meshing only needs `{gid: Polygon | MultiPolygon}`. Requires `gmsh` (`pip install upxo[mesh]`).

## 1. Monte-Carlo grain structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from upxo.ggrowth.mcgs import mcgs
from upxo.pxtal.geometrification import GrainManifold2D
import upxo.gsdataops.grid_ops as gridOps
from upxo.viz import gsviz
from upxo.meshing.gsmesh2d import mesh_gs, visualize_gs_mesh
from upxo.meshing.writer_ABQ import summarize_inp
from pathlib import Path
_XLS = next(
    (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls").is_file()),
    Path("confMesh1.xls"),
)
print("dashboard:", _XLS)

In [ ]:
pxt = mcgs(input_dashboard=str(_XLS))
pxt.simulate()
pxt.detect_grains(library="cc3d", connectivity=4)
gstslice = pxt.gs[list(pxt.gs.keys())[-1]]
lfi = gstslice.lfi
print("LFI", lfi.shape, "grains", len(np.unique(lfi)) - 1)
plt.imshow(lfi, origin="lower")
plt.title("LFI")
plt.gca().set_aspect("equal")

## 2. Hybrid seeds and Voronoi manifold

Dense seeds near GBs, coarser in the bulk. `GrainManifold2D.by_tessellation` unions Voronoi cells that share a label.

In [ ]:
seeds = gridOps.generate_constrained_hybrid_seeds(
    lfi, target_spacing=1.0, bulk_spacing=1.0,
    jitter_factor=0.25, margin=0.5, padding=2.0)
manifold = GrainManifold2D.by_tessellation(lfi, seeds)
original_cells = {gid: geom for gid, geom in manifold.cells.items()}
print("grains in manifold", len(manifold.cells), "seeds", len(seeds))
gsviz.plot_manifold_geom([original_cells], figsize=(6, 6), dpi=110, inlude_legend=False)

## 3. Smooth interfaces and trim to the RVE

In [ ]:
manifold.smooth_interfaces(iterations=10, lmbda=0.5, mu=-0.53)
h, w = lfi.shape[:2]
pad = max(2, min(h, w) // 10)
_ = manifold.trim_to_rve(bounds=(pad, pad, w - pad, h - pad))
gsviz.plot_manifold_geom(
    [original_cells, manifold.cells],
    figsize=(12, 6), dpi=120, inlude_legend=False)

## 4. Conformal mesh, plot, Abaqus export

`mesh_gs` skips empty/zero-area parts. Plot and INP go through the same library APIs as `confMesh2d_gmsh.ipynb`.

In [ ]:
result = mesh_gs(
    manifold.cells, mesh_size_gb=1.5, mesh_size_bulk=3.0,
    mesh_algo=6, recombine_to_quads=False, verbose=True)
m = result["mesher"]
m.form_elsets_gmsh()
m.build_boundary_nsets()
m.build_gb_nset()
print(m.validation_report)
fig, ax = visualize_gs_mesh(result, figsize=(6, 6), dpi=120, show_nsets=True)
fig

In [ ]:
out = Path.cwd() / "confMesh2d_technique_b_out"
inp = m.export_abaqus_inp(out / "rve_cps3.inp", plane="stress")
inp, summarize_inp(inp)